# Vem Subclasses, Over the Full Timecourse (Cell + Nucleus)

Same shared embedding, with a kernel density overlay per subclass
(dying, persisting, slow proliferator, fast proliferator) at each
frame - every available frame, not just the first 13 the way
`naive_vs_vem`'s own `03_vem_subclasses_over_time.ipynb` is capped. This
analysis only ever covers `vem_1uM` (no `naive`), so there's no
condition filtering needed here - everything in the embedding already
is vem_1uM.

The hypothesis this notebook tests: `naive_vs_vem`'s subclasses don't
separate clearly within the first 13 frames - maybe they do separate,
just later.

Plots both embeddings `main.py` computes (UMAP and PHATE) every time
this notebook runs, no manual switch needed.

Requires `main.py` to have been run first.


In [ ]:
import os
import sys

import matplotlib.pyplot as plt

# assumes Jupyter's cwd is this notebook's own directory
# (analyses/04_vem_full_timecourse/notebooks/), so the analysis dir -
# and config.py - is one level up, and the repo root - where shared/
# lives - is two more levels up from there
NOTEBOOK_DIR = os.getcwd()
ANALYSIS_DIR = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..'))
REPO_ROOT    = os.path.abspath(os.path.join(ANALYSIS_DIR, '..', '..'))
sys.path.insert(0, REPO_ROOT)
sys.path.insert(0, ANALYSIS_DIR)
import config

# config.py's OUTPUT_DIR/CAJAL_OUTPUT_DIR are relative to the analysis
# directory, not this notebook's own cwd - re-anchor them to an absolute
# path once, here
config.OUTPUT_DIR       = os.path.join(ANALYSIS_DIR, config.OUTPUT_DIR)
config.CAJAL_OUTPUT_DIR = os.path.join(ANALYSIS_DIR, config.CAJAL_OUTPUT_DIR)

from shared.load_data import load_full_dataset, export_plot_data
from shared.density_plots import export_kde_by_frame


## Plot: embedding with per-subclass kernel density, every frame

For each embedding method, one PDF, one SVG per frame, written to
`output/plots/01_subclasses_over_time/` (SVGs in their own `svgs/`
subfolder there).


In [ ]:
SUBCLASS_COLORS = {
    'dying':       'tab:red',
    'persisting':  'tab:purple',
    'slow_prolif': 'tab:orange',
    'fast_prolif': 'tab:green',
}
NOTEBOOK_PLOTS_DIR = os.path.join(config.OUTPUT_DIR, 'plots', '01_subclasses_over_time')
MODALITY_NAMES = {'cell': 'cell', 'nuc': 'nucleus'}

# KDE contour levels (level p encloses the densest (1-p) fraction of a
# group's mass - higher is tighter/closer to the peak) and bandwidth
# multiplier (above 1 smooths over sampling/distance-tie noise)
KDE_LEVELS     = (0.7, 0.8, 0.9)
KDE_BW_ADJUST  = 1

# main.py computes both of these for every modality - plot both every
# time this notebook runs, rather than picking just one
EMBEDDING_METHODS = ['umap', 'phate']


def process_modality(modality):
    '''
    For each embedding method, load this modality's embedding and
    metadata and export a PDF and a per-frame SVG with a KDE contour per
    subclass, across every available frame.

    Inputs:
        modality: 'cell' or 'nuc'
    Outputs:
        dfs: {embedding_method: the full dataset for this modality and
              embedding method} (in-memory only)
    '''
    dfs = {}
    for embedding_method in EMBEDDING_METHODS:
        df = load_full_dataset(config.CAJAL_OUTPUT_DIR, modality, embedding_method)
        print(f"[{modality}/{embedding_method}] {len(df)} cells across "
              f"{df['subclass'].nunique()} subclasses, {df['frame'].nunique()} frames "
              f"(frames {df['frame'].min()}-{df['frame'].max()})")

        export_kde_by_frame(df, 'subclass', SUBCLASS_COLORS, NOTEBOOK_PLOTS_DIR,
                             f'{modality}_{embedding_method}', embedding_method.upper(),
                             title_prefix=f'{MODALITY_NAMES[modality]} morphology (subclasses)',
                             levels=KDE_LEVELS, bw_adjust=KDE_BW_ADJUST)
        dfs[embedding_method] = df

    return dfs


## Run: cell morphology

In [ ]:
cell_dfs = process_modality('cell')


## Run: nuclear morphology

In [ ]:
nuc_dfs = process_modality('nuc')


## Export plot data

Combines every (modality, embedding method) DataFrame above into one CSV
(`plot_data.csv`, alongside the PDFs/SVGs above) - the same dim_0/dim_1
coordinates and metadata the KDE plots above use, so someone without
this repo can recreate or extend these plots from the raw data alone.

In [ ]:
plot_data = {('cell', m): df for m, df in cell_dfs.items()}
plot_data.update({('nuc', m): df for m, df in nuc_dfs.items()})
export_plot_data(plot_data, os.path.join(NOTEBOOK_PLOTS_DIR, 'plot_data.csv'))
